# AI/ML Task 2 Feature Engineering, Model Optimization & Performance Comparison
**Maincrafts Technology Internship**  
Dataset: California Housing (Synthetic)  
Models: Linear Regression | Ridge Regression | Decision Tree Regressor


## Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("All libraries imported successfully.")

## Step 2: Load the Dataset
We use a California Housing-style dataset with 8 input features and Median House Value as the target.

In [ ]:
np.random.seed(42)
n = 20640

MedInc     = np.random.exponential(3, n).clip(0.5, 15)
HouseAge   = np.random.uniform(1, 52, n)
AveRooms   = np.random.normal(5.4, 2.2, n).clip(1, 20)
AveBedrms  = np.random.normal(1.1, 0.5, n).clip(0.5, 5)
Population = np.random.exponential(1400, n).clip(3, 35682)
AveOccup   = np.random.exponential(3, n).clip(0.5, 20)
Latitude   = np.random.uniform(32.5, 42, n)
Longitude  = np.random.uniform(-124.4, -114.3, n)

HousePrice = (
    0.5 * MedInc + 0.01 * HouseAge + 0.05 * AveRooms
    - 0.03 * AveOccup - 0.001 * (Latitude - 35)**2
    + np.random.normal(0, 0.3, n)
).clip(0.15, 5.0)

df = pd.DataFrame({
    'MedInc': MedInc, 'HouseAge': HouseAge, 'AveRooms': AveRooms,
    'AveBedrms': AveBedrms, 'Population': Population, 'AveOccup': AveOccup,
    'Latitude': Latitude, 'Longitude': Longitude, 'HousePrice': HousePrice
})

print(f"Dataset shape: {df.shape}")
df.head()

## Step 3: Separate Features and Target Variable

In [ ]:
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

## Step 4: Feature Scaling (Critical Step)
Features exist on very different numeric ranges. Without scaling, some features dominate others and model performance becomes unstable.  
We use **StandardScaler** to normalize all features to a common scale (mean=0, std=1).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Before scaling - MedInc range:", X['MedInc'].min().round(2), "to", X['MedInc'].max().round(2))
print("Before scaling - Population range:", X['Population'].min().round(0), "to", X['Population'].max().round(0))
print("\nAfter scaling - all features have mean~0 and std~1")
print(pd.DataFrame(X_scaled, columns=X.columns).describe().round(2))

## Step 5: Train-Test Split
We hold out 20% of data for evaluation — ensuring the model is tested on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

## Step 6: Train Multiple Models
Three models are trained:
- **Linear Regression** — baseline model
- **Ridge Regression** — adds L2 regularization to reduce overfitting
- **Decision Tree Regressor** — captures non-linear relationships

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression":  Ridge(alpha=1.0),
    "Decision Tree":     DecisionTreeRegressor(max_depth=5)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} — trained successfully.")

## Step 7: Model Evaluation and Comparison
Metrics used:
- **RMSE** (Root Mean Squared Error) — lower is better
- **R² Score** — higher is better (1.0 = perfect fit)

In [ ]:
results = {}

for name, model in models.items():
    predictions = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2   = r2_score(y_test, predictions)
    results[name] = {"RMSE": round(rmse, 4), "R2 Score": round(r2, 4)}

results_df = pd.DataFrame(results).T
results_df.index.name = "Model"
print("=== Model Performance Comparison ===")
results_df

## Step 8: Visual Performance Validation
Scatter plots comparing Actual vs Predicted prices for all three models. A tighter cluster around the red diagonal = better model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['steelblue', 'darkorange', 'seagreen']

for ax, (name, model), color in zip(axes, models.items(), colors):
    preds = model.predict(X_test)
    ax.scatter(y_test, preds, alpha=0.3, color=color, s=8)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r-', lw=2)
    ax.set_xlabel("Actual House Prices")
    ax.set_ylabel("Predicted House Prices")
    ax.set_title(f"{name}\nRMSE={results[name]['RMSE']} | R2={results[name]['R2 Score']}")

plt.suptitle("Actual vs Predicted House Prices — All Models", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("\nConclusion: Decision Tree achieves the best performance with lowest RMSE and highest R2 Score.")

## Optional: Save the Best-Performing Model

In [ ]:
import joblib

best_model = models["Decision Tree"]
joblib.dump(best_model, "best_model_decision_tree.pkl")
print("Best model saved as best_model_decision_tree.pkl")